# Participant Agent Interaction: complete running code

This notebook exposes the full20-pair interaction procedure. Independent pairs run in parallel; each pair advances one reviewed turn at a time. It includes the entire source for input construction, GPU transport, stopping reviews and export. The default run makes no model calls.

## Environment and materials

Restore the Mistral weights and participant adapters, then follow `Parallel Agent Setup/README.md`. The full assigned article is passed to each participant. New transcripts are saved separately from the frozen conversations used by the manager experiments.

In [ ]:
from pathlib import Path
import sys, json, hashlib, importlib.util
YOUR_PATHWAY = "YOUR PATHWAY"
if YOUR_PATHWAY == "YOUR PATHWAY":
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Models/adapter_manifest.json").is_file()), None)
    if ROOT is None:
        raise ValueError("Set YOUR_PATHWAY to the repository clone directory.")
else:
    ROOT = Path(YOUR_PATHWAY).expanduser().resolve()
sys.path[:0] = [str(ROOT / "Experiment Codes/src"), str(ROOT / "Parallel Agent Setup/src")]
RUN_INFERENCE = False
WITH_MANIPULATION_CHECKS = False


## Complete source used by this notebook

The source below is shown verbatim for review. Execution imports these same files from this repository; the following check rejects a stale notebook snapshot. The source is not copied into an independent implementation.

In [ ]:
SOURCE_SHA256 = {'Experiment Codes/src/model_tokenization.py': 'd761ccbab2456b6eb5073f6c3b436fc5b9066f475ffbf7293e54c6eb01f0a7ce', 'Experiment Codes/src/workbench_protocol.py': 'b93541145fe9ccc1e1d0716d0cb301d318aea95e1fba00b712c5185d63e737f7', 'Parallel Agent Setup/src/transport_core.py': 'f2d670c9e9cd6624945afca84fe9f384c58e9de7a19be756813f99c64d395837', 'Parallel Agent Setup/src/workbench_runtime.py': 'f80e133f4f990e26ef88034691e047e3db6b8409913c973a33d15867946b0603', 'Experiment Codes/src/participant_agent_interaction.py': '5f3e9954eaf1d3fa1bfd2d8a0bba200cac7b3adb6afed87caefad60f5a67b5a4', 'Experiment Codes/Mistral/Participant_Agent_Interaction/run_interactions.py': 'd3ff044db4cc96c815c6b044d080946b3ec0f7773f762c693559ee081e5b3338'}
for relative, expected in SOURCE_SHA256.items():
    assert hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == expected, "Source changed: " + relative
print("Verified complete source files:", len(SOURCE_SHA256))

### Experiment Codes/src/model_tokenization.py

```python
"""Exact local Mistral template and pinned Qwen ChatML; CPU only."""
from pathlib import Path
import ast
import base64
import json
import unicodedata

class NativeTokenizer:
    def __init__(self, root):
        root=Path(root)
        config=json.loads((root/'config/model.json').read_text())
        self.family=config['family']
        folder=(root/config['base_model_dir']).resolve()
        if self.family=='Mistral':
            from tokenizers import Tokenizer
            from jinja2 import Environment, StrictUndefined
            self.encoder=Tokenizer.from_file(str(folder/'tokenizer.json'))
            self.config=json.loads((folder/'tokenizer_config.json').read_text())
            self.template=Environment(undefined=StrictUndefined).from_string(self.config['chat_template'])
        elif self.family=='Qwen':
            import tiktoken
            source=ast.parse((folder/'tokenization_qwen.py').read_text())
            pat=next(ast.literal_eval(n.value) for n in source.body if isinstance(n,ast.Assign)
                     and any(isinstance(t,ast.Name) and t.id=='PAT_STR' for t in n.targets))
            ranks={base64.b64decode(token):int(rank) for line in (folder/'qwen.tiktoken').read_bytes().splitlines()
                   if line for token,rank in [line.split()]}
            special={text:151643+i for i,text in enumerate(['<|endoftext|>','<|im_start|>','<|im_end|>']+
                     [f'<|extra_{i}|>' for i in range(205)])}
            self.encoder=tiktoken.Encoding('pinned_qwen',pat_str=pat,mergeable_ranks=ranks,special_tokens=special)
        else: raise ValueError('Unknown model family')

    def prepare(self, content):
        if self.family=='Mistral':
            def invalid(message):raise ValueError(message)
            text=self.template.render(messages=[{'role':'user','content':content}],
                 bos_token=self.config['bos_token'],eos_token=self.config['eos_token'],
                 raise_exception=invalid,add_generation_prompt=True)
            return text,self.encoder.encode(text,add_special_tokens=False).ids
        encode=lambda text:self.encoder.encode(unicodedata.normalize('NFC',text),allowed_special=set(),disallowed_special=())
        nl=encode('\n')
        ids=([151644]+encode('system')+nl+encode('You are a helpful assistant.')+[151645]+nl+
             [151644]+encode('user')+nl+encode(content)+[151645]+nl+[151644]+encode('assistant')+nl)
        return self.encoder.decode(ids),ids
```

### Experiment Codes/src/workbench_protocol.py

```python
"""Pure input construction: no network, GPU, model generation, or prior responses."""
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import hashlib
import json
import random
import re
import uuid

ROOT = Path(__file__).resolve().parents[1] / "Mistral"
TRANSCRIPT_INTRO = 'The following is a conversation between two other participants. You did not take part in it.'
OWN_ANSWER_INSTRUCTION = 'Before reviewing, state your own answer to the question.'
NEUTRAL = {'P1': 'P1', 'P2': 'P2'}


def read_json(path):
    return json.loads(Path(path).read_text())


def sha_text(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.part')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n')
    temp.replace(path)


def write_json_once(path, value):
    """Create immutable run provenance, or require an identical existing file."""
    path = Path(path)
    data = json.dumps(value, ensure_ascii=False, indent=2) + '\n'
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with path.open('x') as file:
            file.write(data)
    except FileExistsError:
        if path.read_text() != data:
            raise ValueError('Existing run provenance differs; use a new approved run ID: ' + str(path))


def bind_run_sources(run_root, source_paths, root=ROOT):
    root = Path(root)
    files = {str(Path(path).relative_to(root)): hashlib.sha256(Path(path).read_bytes()).hexdigest()
             for path in sorted(map(Path, source_paths))}
    write_json_once(Path(run_root)/'inputs/source_hashes.json', files)
    return files


def seed_for(identifier, base=20261018):
    return (base + int(sha_text(identifier)[:8], 16)) % 2**32


@dataclass(frozen=True)
class AgentInstance:
    """A distinct request context sharing read-only base/adapter tensors.

    There is no mutable conversation, optimizer, or cross-row cache here. Sharing
    a frozen adapter does not share a conversation or update anyone's weights.
    """
    instance_id: str
    adapter_alias: str
    seed: int
    history: tuple = ()


def spawn_instance(key, adapter_alias):
    return AgentInstance(str(uuid.uuid5(uuid.NAMESPACE_URL, 'parallel-workbench:' + key)),
                         adapter_alias, seed_for(key))


def spawn_original_agents(run_id='participant_prepared'):
    return {a: spawn_instance(run_id + ':original:' + a, 'participant_' + a.lower()) for a in 'ABCDE'}


def neutralize(text, actual_to_neutral):
    """Rename explicit speakers once, never by chained replacements.

    Do not replace bare single letters: 'A' may be a word, scientific label or
    article content. Frozen Participant Agent Interaction speaker references use 'Agent A', etc.
    """
    def replace(match):
        actual = match.group(1)
        if actual not in actual_to_neutral:
            raise ValueError('Transcript references an agent outside the pair: ' + actual)
        return actual_to_neutral[actual]
    return re.sub(r'\bAgent\s+([ABCDE])\b', replace, text)



def experiment_number(experiment):
    """Resolve the procedure number from a numeric selector or manuscript ID."""
    match = re.fullmatch(r'0?([123])([ab]?)', str(experiment).strip().lower())
    if not match:
        raise ValueError('Unknown experiment: ' + str(experiment))
    return int(match.group(1))


def experiment_id(experiment, root=ROOT):
    """Use a for Mistral and b for Qwen; reject a mismatched family."""
    number = experiment_number(experiment)
    family = read_json(Path(root)/'config/model.json')['family']
    suffix = {'Mistral': 'a', 'Qwen': 'b'}[family]
    value = str(experiment).strip().lower()
    if value[-1:] in ('a', 'b') and value[-1] != suffix:
        raise ValueError(f'Experiment {value} does not use the {family} model family')
    return f'{number}{suffix}'


def experiment_directory(experiment, root=ROOT):
    return Path(root)/f'Experiment_{experiment_id(experiment, root)}'


def validate_base_prompt(baseline, experiment):
    """Reject incomplete prompt copies before preparing or sending any request."""
    choice_field = 'VERDICT:' if experiment_number(experiment) == 3 else 'ENDORSEMENT:'
    if not any(label in baseline for label in ('YOUR OWN ANSWER:', 'YOUR OWN ANSER:')):
        raise ValueError('Incomplete base prompt: missing own-answer output field')
    if choice_field not in baseline or 'RATIONALE:' not in baseline:
        raise ValueError('Incomplete base prompt: missing choice or rationale output field')
    if 'Return only these three fields.' not in baseline:
        raise ValueError('Incomplete base prompt: missing closing output instruction')
    if experiment_number(experiment) == 3 and not all(label in baseline for label in ('P1:', 'P2:', 'Both:', 'None:')):
        raise ValueError('Incomplete verdict prompt: missing category definitions')
    return True


def build_cases(experiment, root=ROOT):
    """100 article/pair/manager cases x two separately generated orientations."""
    root = Path(root)
    experiment = experiment_id(experiment, root)
    number = experiment_number(experiment)
    baseline = (root/f'Experiment_{experiment}/prompts/Prompt_Verbatim.txt').read_text()
    validate_base_prompt(baseline, experiment)
    model_config = read_json(root/'config/model.json')
    sessions = read_json(root/'inputs/frozen_sessions.json')
    differences = {(d['article_id'], d['manager'], d['participant']): d
                   for d in read_json(root/'inputs/belief_differences.json')}
    cases = []
    for session in sessions:
        for manager in 'ABCDE':
            base = f'{session["session_id"]}_manager_{manager}'
            for orientation in ['original', 'swapped']:
                pair = session['agents']
                actuals = pair if orientation == 'original' else list(reversed(pair))
                labels = dict(zip(['P1','P2'], actuals))
                actual_to_neutral = {actual: NEUTRAL[p] for p, actual in labels.items()}
                # Keep the established numeric namespace so renaming preserves seeds and UUIDs.
                row_id = f'{model_config["family"].lower()}_a{number:02d}_{base}_{orientation}'
                instance = spawn_instance(row_id, model_config['manager_alias_prefix'] + manager.lower())
                difference_texts = {
                    p: differences[(session['article_id'], manager, actual)]['text'].format(participant=NEUTRAL[p])
                    for p, actual in labels.items()}
                messages = [{**m, 'speaker': actual_to_neutral[m['actual_agent']],
                             'text': neutralize(m['text'], actual_to_neutral)} for m in session['messages']]
                finals = {p: session['final_message_ids'][actual] for p, actual in labels.items()}
                allowed = ['P1', 'P2'] if number != 3 else (
                    ['P1','P2','None'] if session['source_outcome'] == 'Unresolved'
                    else ['P1','P2','Both','None'])
                # Only substitute the explicitly requested manager placeholder.
                # Its neutral role name does not reveal the participant's letter.
                prompt = baseline.replace('{manager}', 'manager')
                sections = [prompt, OWN_ANSWER_INSTRUCTION,
                    'FACTUAL QUESTION\n' + session['question'],
                    'PARTICIPANT IDENTITIES FOR THIS EVALUATION\nP1 and P2 identify the two participants consistently in every supplied material.',
                    'BELIEF DIFFERENCES RELATIVE TO MY LEARNED BELIEFS\n' +
                    '\n'.join(f'Beliefs of {NEUTRAL[p]} ({p}): {difference_texts[p]}' for p in ['P1','P2']),
                    'FINAL MESSAGE IDS\n' + '\n'.join(f'{p} ({NEUTRAL[p]}): {finals[p]}' for p in ['P1','P2'])]
                if number in [2, 3]:
                    transcript = '\n\n'.join(f'{m["id"]} | {m["speaker"]} | {m["stage"]}\n{m["text"]}' for m in messages)
                    sections.append(TRANSCRIPT_INTRO + '\n<OTHER_PARTICIPANTS_CONVERSATION>\n' +
                                    transcript + '\n</OTHER_PARTICIPANTS_CONVERSATION>')
                else:
                    sections.append('No interaction transcript is supplied in this attempt. The final message IDs identify the participants\' final messages; their contents are not supplied.')
                if number == 3:
                    sections.append('PERMITTED VERDICTS\n' + ', '.join(allowed))
                content = '\n\n'.join(sections)
                cases.append({'row_id': row_id, 'base_row_id': base, 'experiment': experiment,
                    'article_id': session['article_id'], 'article_title': session['article_title'],
                    'session_id': session['session_id'], 'pair': session['pair'], 'manager': manager,
                    'orientation': orientation, 'labels_to_actual': labels, 'neutral_to_actual':
                        {NEUTRAL[p]: a for p, a in labels.items()},
                    'agent_instance': asdict(instance), 'question': session['question'],
                    'components': session['components'], 'belief_differences': difference_texts,
                    'final_message_ids': finals, 'messages': messages, 'allowed_choices': allowed,
                    'source_outcome_offline_only': session['source_outcome'],
                    'baseline_prompt_sha256': sha_text(baseline), 'content': content,
                    'content_sha256': sha_text(content),
                    'source_session_sha256': session['source_session_sha256'],
                    'own_answer_measurement': 'same-call pre-decision answer; exposed to all input materials'})
    validate_cases(cases)
    return cases


def validate_cases(cases):
    if len(cases) != 200 or len({c['row_id'] for c in cases}) != 200:
        raise ValueError('Exactly 200 independently identified calls are required')
    if len({c['agent_instance']['instance_id'] for c in cases}) != 200:
        raise ValueError('Manager contexts must be unique')
    if len({c['agent_instance']['seed'] for c in cases}) != 200:
        raise ValueError('Request seeds unexpectedly collide')
    grouped = {}
    for c in cases:
        if c['agent_instance']['history'] not in [(), []]:
            raise ValueError('Manager context must start empty')
        grouped.setdefault(c['base_row_id'], []).append(c)
        if OWN_ANSWER_INSTRUCTION not in c['content']:
            raise ValueError('Missing own-answer instruction')
        if re.search(r'\bAgent\s+[ABCDE]\b|Manager_Agent_Pair_|P[12]\s*=\s*[ABCDE]\b', c['content']):
            raise ValueError('Original agent identity leaked into manager prompt')
        if experiment_number(c['experiment']) in [2, 3] and TRANSCRIPT_INTRO not in c['content']:
            raise ValueError('Missing third-party transcript frame')
        for p, mid in c['final_message_ids'].items():
            matches = [m for m in c['messages'] if m['id'] == mid]
            if len(matches) != 1 or matches[0]['actual_agent'] != c['labels_to_actual'][p]:
                raise ValueError('Final-message attribution mismatch')
    if len(grouped) != 100:
        raise ValueError('Expected 100 matched cases')
    for pair in grouped.values():
        if len(pair) != 2 or {c['orientation'] for c in pair} != {'original','swapped'}:
            raise ValueError('Missing independently requested orientation')
        a, b = sorted(pair, key=lambda c:c['orientation'])
        if a['labels_to_actual']['P1'] != b['labels_to_actual']['P2']:
            raise ValueError('Participant positions were not exchanged')
        if a['labels_to_actual']['P2'] != b['labels_to_actual']['P1']:
            raise ValueError('Participant positions were not exchanged')
    return True


def randomized_execution_order(cases, seed=20261018):
    """Random scheduling order; reporting retains adjacent matched orientations."""
    rows = list(cases)
    random.Random(seed).shuffle(rows)
    return rows


class FrozenTokenizer:
    """CPU-only tokenizer; exact local base vocabulary, no weight download."""
    def __init__(self, root=ROOT):
        from model_tokenization import NativeTokenizer
        self.impl = NativeTokenizer(root)
    def serialize(self, content):
        return self.impl.prepare(content)[0]
    def prepare(self, content):
        return self.impl.prepare(content)


def prepare_inputs(experiment, root=ROOT, save=True):
    """Author 200 exact input strings and token IDs. Never generate responses."""
    root = Path(root)
    cases = build_cases(experiment, root)
    tokenizer = FrozenTokenizer(root)
    config = read_json(experiment_directory(experiment, root)/'config.json')
    for c in cases:
        c['serialized'], c['input_ids'] = tokenizer.prepare(c['content'])
        c['input_tokens'] = len(c['input_ids'])
        if c['input_tokens'] + config['max_output_tokens'] > read_json(root/'config/model.json')['context_tokens']:
            raise ValueError('Context overflow; inputs must not be truncated: ' + c['row_id'])
    if save:
        dest = experiment_directory(experiment, root)/'inputs'
        write_json(dest/'prepared_cases.json', cases)
        write_json(dest/'all_input_texts.json', [c['serialized'] for c in cases])
        write_json(dest/'all_input_token_ids.json', [c['input_ids'] for c in cases])
        write_json(dest/'row_order.json', [c['row_id'] for c in cases])
        write_json(dest/'execution_order.json', [c['row_id'] for c in randomized_execution_order(cases)])
    return cases
```

### Parallel Agent Setup/src/transport_core.py

```python
"""Isolated, resumable requests to one immutable multi-LoRA vLLM server.

This is infrastructure code. It contains no experiment-specific choice rules,
article materials, endorsement policies, or repair loops.
"""
from __future__ import annotations
import asyncio
import codecs
import datetime as dt
import hashlib
import json
import os
import time
from pathlib import Path
from typing import Any
import httpx

def stamp():
    return dt.datetime.now(dt.timezone.utc).isoformat()

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def atomic_json(path, value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.part')
    tmp.write_text(json.dumps(value,indent=2,ensure_ascii=False)+'\n')
    tmp.replace(path)

def runtime_digest(root):
    root=Path(root)
    server=json.loads((root/'config/server_process.json').read_text())
    return digest({'server_command':server['command'],
      'environment_lock':(root/'config/requirements.lock.txt').read_text(),
      'base_model_manifest':json.loads((root/'config/base_model_manifest.json').read_text()),
      'client_source_sha256':hashlib.sha256(Path(__file__).read_bytes()).hexdigest()})

def request_digest(payload, alias, runtime):
    return digest({'payload':payload,'adapter':alias,'runtime':runtime})

async def journaled_lines(response, path):
    decoder=codecs.getincrementaldecoder('utf-8')()
    pending=''
    with Path(path).open('ab') as journal:
        async for chunk in response.aiter_bytes():
            journal.write(chunk);journal.flush()
            pending+=decoder.decode(chunk)
            while '\n' in pending:
                line,pending=pending.split('\n',1)
                yield line.removesuffix('\r')
        pending+=decoder.decode(b'',final=True)
        if pending:yield pending.removesuffix('\r')

def validate_job(job, aliases, context_limit=16384):
    if job['model'] not in aliases:raise ValueError('Unregistered adapter: '+job['model'])
    ids=job['input_ids']
    if not ids or any(type(x) is not int or x<0 for x in ids):raise ValueError('Explicit token IDs required')
    if len(ids)+job['max_tokens']>context_limit:raise ValueError('Context budget exceeded; truncation is forbidden')
    if not 0<=job['temperature']<=2:raise ValueError('Invalid temperature')
    if not 0<=job['seed']<2**32:raise ValueError('Explicit 32-bit request seed required')
    if not job['call_id'].replace('_','').replace('-','').isalnum():raise ValueError('Unsafe call ID')
    return {
      'model':job['model'],'prompt':ids,'temperature':job['temperature'],
      'max_tokens':job['max_tokens'],'seed':job['seed'],'n':1,
      'top_p':1.0,'top_k':-1,'min_p':0.0,'repetition_penalty':1.0,
      'presence_penalty':0.0,'frequency_penalty':0.0,
      'stop_token_ids':job.get('stop_token_ids',[]),'stop':job.get('stop',[]),'include_stop_str_in_output':True,
      'skip_special_tokens':False,'logprobs':5,'return_tokens_as_token_ids':True,'return_token_ids':True,
      'min_tokens':job.get('min_tokens',0),
      'stream':True,'stream_options':{'include_usage':True}}

async def execute_one(client, semaphore, job, root, aliases, endpoint):
    payload=validate_job(job, aliases)
    runtime=runtime_digest(root)
    signature=request_digest(payload,aliases[job['model']],runtime)
    path=Path(root)/'logs/calls'/(job['call_id']+'.json')
    if path.exists():
        saved=json.loads(path.read_text())
        if saved['request_sha256']!=signature:raise ValueError('A saved call ID has different inputs')
        if saved['status']=='complete':return {**saved,'resumed_from_disk':True}
        raise RuntimeError('Previous call failed; explicit new run ID required: '+job['call_id'])
    path.parent.mkdir(parents=True,exist_ok=True)
    claim=path.with_suffix('.claim')
    fd=os.open(claim,os.O_CREAT|os.O_EXCL|os.O_WRONLY,0o600)
    os.write(fd,str(os.getpid()).encode());os.close(fd)
    queued=time.perf_counter()
    record={'call_id':job['call_id'],'kind':job.get('kind', 'research_call'),
      'status':'queued','queued_utc':stamp(),'request_sha256':signature,
      'request':payload,'adapter':aliases[job['model']],'runtime_sha256':runtime,
      'serialized_input':job['serialized'],'input_tokens':len(job['input_ids']),
      'output_text':'','output_token_ids':[],'logprob_token_ids':[],
      'server_prompt_token_ids':None,'events':[],'usage':None,'received_done':False,
      'finish_reason':None,'stop_reason':None}
    atomic_json(path,record)
    async with semaphore:
        start=time.perf_counter();record.update(status='running',started_utc=stamp(),client_queue_seconds=start-queued)
        atomic_json(path,record)
        print('REQUEST_START', job['call_id'], 'INPUT_TOKENS', len(job['input_ids']), flush=True)
        first=None
        try:
            async with client.stream('POST',endpoint+'/v1/completions',json=payload) as response:
                if response.status_code!=200:
                    await response.aread()
                    raise RuntimeError(f'HTTP {response.status_code}: {response.text[:2000]}')
                async for line in journaled_lines(response,path.with_suffix('.sse')):
                    if not line.startswith('data:'):continue
                    raw=line[5:].lstrip()
                    if raw=='[DONE]':
                        record['received_done']=True;break
                    event=json.loads(raw);record['events'].append(event)
                    if event.get('error'):raise RuntimeError(str(event['error']))
                    if event.get('usage') is not None:record['usage']=event['usage']
                    for choice in event.get('choices',[]):
                        if choice.get('prompt_token_ids') is not None:
                            if record['server_prompt_token_ids'] is not None and record['server_prompt_token_ids']!=choice['prompt_token_ids']:
                                raise RuntimeError('Server prompt token IDs changed within one request')
                            record['server_prompt_token_ids']=choice['prompt_token_ids']
                        record['output_token_ids'].extend(choice.get('token_ids') or [])
                        text=choice.get('text','')
                        if text and first is None:first=time.perf_counter()
                        record['output_text']+=text
                        tokens=(choice.get('logprobs') or {}).get('tokens') or []
                        for token in tokens:
                            if token.startswith('token_id:'):
                                record['logprob_token_ids'].append(int(token.split(':',1)[1]))
                        if choice.get('finish_reason') is not None:record['finish_reason']=choice['finish_reason']
                        if choice.get('stop_reason') is not None:record['stop_reason']=choice['stop_reason']
            if record['usage'] is None:raise RuntimeError('Missing server token usage')
            if record['usage']['prompt_tokens']!=record['input_tokens']:raise RuntimeError('Server input token count mismatch')
            if record['server_prompt_token_ids']!=job['input_ids']:raise RuntimeError('Server prompt token IDs mismatch')
            if not record['received_done'] or record['finish_reason'] not in ['stop','length']:raise RuntimeError('Incomplete completion stream')
            record['output_tokens']=record['usage']['completion_tokens']
            record['output_ids_match_usage']=len(record['output_token_ids'])==record['output_tokens']
            if not record['output_ids_match_usage']:raise RuntimeError('Output token IDs/usage mismatch')
            if record['logprob_token_ids']!=record['output_token_ids']:raise RuntimeError('Logprob and output token IDs mismatch')
            record['status']='complete'
        except asyncio.CancelledError:
            record.update(status='interrupted',error='Request task cancelled; explicit recovery is required')
            raise
        except Exception as exc:
            record.update(status='failed',error=repr(exc))
        finally:
            end=time.perf_counter()
            record.update(finished_utc=stamp(),elapsed_seconds=end-start,
              total_seconds=end-queued,time_to_first_text_seconds=None if first is None else first-start)
            atomic_json(path,record)
            claim.unlink(missing_ok=True)
            print('REQUEST',job['call_id'],record['status'],'INPUT',record['input_tokens'],
                  'OUTPUT',record.get('output_tokens'),'QUEUE_SECONDS',round(record['client_queue_seconds'],3),
                  'GENERATION_SECONDS',round(record['elapsed_seconds'],3),
                  'TOTAL_SECONDS',round(record['total_seconds'],3),flush=True)
    return record

async def monitor_memory(path, stop):
    with Path(path).open('a') as file:
        while not stop.is_set():
            process=await asyncio.create_subprocess_exec('nvidia-smi',
              '--query-gpu=memory.used,memory.free,utilization.gpu','--format=csv,noheader,nounits',
              stdout=asyncio.subprocess.PIPE,stderr=asyncio.subprocess.PIPE)
            out,err=await process.communicate()
            sample={'utc':stamp(),'gpu':out.decode().strip(),'error':err.decode().strip()}
            sample['meminfo']=dict(line.split(':',1) for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith(('MemAvailable:','MemTotal:')))
            file.write(json.dumps(sample)+'\n');file.flush()
            try:await asyncio.wait_for(stop.wait(),timeout=1.0)
            except asyncio.TimeoutError:pass

async def run_batch(jobs, concurrency, root, run_name, aliases, endpoint='http://127.0.0.1:8000'):
    invocation_start=time.perf_counter()
    root=Path(root)
    if len({j['call_id'] for j in jobs})!=len(jobs):raise ValueError('Duplicate call IDs')
    for job in jobs:validate_job(job,aliases)
    runtime=runtime_digest(root)
    for job in jobs:
        path=root/'logs/calls'/(job['call_id']+'.json')
        if path.with_suffix('.claim').exists():raise RuntimeError('Call already claimed; inspect before recovery')
        if path.exists():
            value=json.loads(path.read_text())
            expected=request_digest(validate_job(job,aliases),aliases[job['model']],runtime)
            if value['request_sha256']!=expected:raise ValueError('Saved input/runtime signature changed')
            if value['status']!='complete':raise RuntimeError('Previous call is incomplete; explicit new diagnostic run ID required')
    summary_path=root/'qa'/f'{run_name}.json'
    if summary_path.exists():
        previous=json.loads(summary_path.read_text())
        if previous.get('failed')==0 and previous.get('call_ids')==[j['call_id'] for j in jobs] and previous.get('concurrency')==concurrency:
            saved=[]
            for job in jobs:
                value=json.loads((root/'logs/calls'/(job['call_id']+'.json')).read_text())
                expected=request_digest(validate_job(job,aliases),aliases[job['model']],runtime)
                if value['request_sha256']!=expected:raise ValueError('Existing batch inputs changed')
                saved.append(value)
            if all(value['status']=='complete' for value in saved):
                print('BATCH_RESUMED_WITHOUT_GENERATION',run_name,flush=True)
                return {**previous,'resumed_without_generation':True,
                    'historical_batch_elapsed_seconds':previous['elapsed_seconds'],
                    'current_invocation_seconds':time.perf_counter()-invocation_start},[
                    {**value,'resumed_from_disk':True} for value in saved]
    stop=asyncio.Event();memory_path=root/'logs'/f'{run_name}_memory.jsonl'
    memory_path.parent.mkdir(parents=True,exist_ok=True)
    monitor=asyncio.create_task(monitor_memory(memory_path,stop))
    start=time.perf_counter()
    try:
        async with httpx.AsyncClient(timeout=httpx.Timeout(600.0,connect=30.0),
          limits=httpx.Limits(max_connections=concurrency,max_keepalive_connections=concurrency),trust_env=False) as client:
            before=await client.get(endpoint+'/metrics')
            (root/'logs'/f'{run_name}_metrics_before.txt').write_text(before.text)
            semaphore=asyncio.Semaphore(concurrency)
            tasks=[asyncio.create_task(execute_one(client,semaphore,j,root,aliases,endpoint)) for j in jobs]
            try:
                results=await asyncio.gather(*tasks,return_exceptions=True)
            except BaseException:
                # Never release the endpoint lock while sibling requests remain active.
                for task in tasks:
                    if not task.done():task.cancel()
                await asyncio.gather(*tasks,return_exceptions=True)
                raise
            errors=[{'call_id':job['call_id'],'error':repr(result)}
                    for job,result in zip(jobs,results) if isinstance(result,BaseException)]
            if errors:
                atomic_json(root/'qa'/f'{run_name}_task_errors.json',errors)
                raise RuntimeError('Request task failures drained safely; inspect the saved task errors')
            after=await client.get(endpoint+'/metrics')
            (root/'logs'/f'{run_name}_metrics_after.txt').write_text(after.text)
    finally:
        stop.set();await monitor
    seconds=time.perf_counter()-start
    complete=[r for r in results if r['status']=='complete']
    used=[r for r in complete if not r.get('resumed_from_disk')]
    summary={'run_name':run_name,'utc':stamp(),'concurrency':concurrency,'requests':len(jobs),
      'completed':len(complete),'failed':len(results)-len(complete),
      'elapsed_seconds':seconds,'input_tokens':sum(r['input_tokens'] for r in complete),
      'output_tokens':sum(r.get('output_tokens',0) for r in complete),
      'fresh_output_tokens_per_second':sum(r.get('output_tokens',0) for r in used)/seconds,
      'all_input_counts_match':all(r.get('usage',{}).get('prompt_tokens')==r['input_tokens'] for r in complete),
      'all_output_ids_match_usage':all(r.get('output_ids_match_usage') for r in complete),
      'call_ids':[j['call_id'] for j in jobs], 'resumed_without_generation':False,
      'current_invocation_seconds':time.perf_counter()-invocation_start}
    atomic_json(root/'qa'/f'{run_name}.json',summary)
    print('BATCH_SUMMARY',json.dumps(summary),flush=True)
    return summary,results
```

### Parallel Agent Setup/src/workbench_runtime.py

```python
"""One immutable multi-LoRA server, bounded concurrency, exact per-call audits."""
from __future__ import annotations
from pathlib import Path
import datetime as dt
import fcntl
import hashlib
import json
import os
import re
import shutil

from workbench_protocol import ROOT, read_json, write_json
from transport_core import run_batch, validate_job

ENDPOINT = 'http://127.0.0.1:8000'
LOCK_PATH = Path(__file__).resolve().parents[2] / '.runtime' / 'api_127_0_0_1_8000.lock'


def require_execution_permission(allow_inference):
    if allow_inference is not True:
        raise PermissionError('Prepared only. Set RUN_INFERENCE=True only after user approval to execute.')


def registry(root=ROOT):
    return {a['alias']: a for a in read_json(Path(root)/'config/adapter_registry.json')}


def choose_admission(jobs, root=ROOT):
    if not jobs:
        raise ValueError('No independent ready requests')
    plan = read_json(Path(root)/'config/recommended_runtime.json')
    aliases = registry(root)
    for job in jobs:
        validate_job(job, aliases, plan['max_context_tokens_including_output'])
    maximum = max(len(j['input_ids']) + j['max_tokens'] for j in jobs)
    if all(len(j['input_ids']) <= 584 and j['max_tokens'] <= 128 for j in jobs):
        profile_cap, basis = 32, 'validated short-context capacity'
    elif all(len(j['input_ids']) <= plan['manager_validated_input_tokens'] and j['max_tokens'] <= 600 for j in jobs):
        profile_cap, basis = plan['manager_validated_concurrency'], 'validated manager profile'
    elif all(len(j['input_ids']) <= plan['participant_validated_input_tokens'] and j['max_tokens'] <= 384 for j in jobs):
        profile_cap, basis = plan['participant_validated_concurrency'], 'validated Participant Agent Interaction profile'
    elif all(j['max_tokens'] <= 600 for j in jobs):
        profile_cap, basis = plan['near_context_limit_validated_concurrency'], 'conservative long-context profile'
    else:
        raise ValueError('Output budget is outside the validated capacity profiles')
    cap = min(len(jobs), profile_cap, plan['server_max_sequences'],
              plan['conservative_total_reserved_tokens'] // maximum)
    if cap < 1:
        raise ValueError('No request fits the token reservation budget')
    cap = min(cap, plan['client_default_concurrency'])
    return {'concurrency': cap, 'maximum_reserved_per_request': maximum,
            'total_reserved_at_cap': cap * maximum, 'profile_cap': profile_cap, 'basis': basis}


def validate_run_id(run_id):
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,70}', run_id):
        raise ValueError('Use a unique run ID with only letters, digits, underscore or hyphen')


def initialize_run(folder, root=ROOT):
    """Copy small immutable runtime metadata; never copy model weights."""
    folder, root = Path(folder), Path(root)
    for d in ['config','logs','qa','data','inputs']:
        (folder/d).mkdir(parents=True, exist_ok=True)
    for name in ['server_process.json','requirements.lock.txt','base_model_manifest.json',
                 'adapter_registry.json','recommended_runtime.json']:
        source = (root/'runtime/server_process.json') if name == 'server_process.json' else root/'config'/name
        target = folder/'config'/name
        if target.exists() and target.read_bytes() != source.read_bytes():
            raise ValueError('Cannot resume a run under changed runtime metadata: ' + name)
        if not target.exists():
            shutil.copy2(source, target)


async def preflight_endpoint(root=ROOT):
    """Read-only health check, called only inside the approved execution path."""
    import httpx
    async with httpx.AsyncClient(timeout=20, trust_env=False) as client:
        health = await client.get(ENDPOINT + '/health')
        health.raise_for_status()
        response = await client.get(ENDPOINT + '/v1/models')
        response.raise_for_status()
        models = response.json()
    available = {x['id'] for x in models['data']}
    expected = set(registry(root))
    if not expected <= available:
        raise RuntimeError('Required adapters are not all registered: ' + repr(expected-available))
    return models


async def run_parallel(jobs, run_name, run_root, *, allow_inference=False, root=ROOT):
    """The only inference gateway; calls inside one batch are independent.

    Existing completed calls are resumed from their exact immutable records.
    Incomplete or changed calls fail loudly; no outcome-based retries exist.
    """
    require_execution_permission(allow_inference)
    validate_run_id(run_name)
    admission = choose_admission(jobs, root)
    initialize_run(run_root, root)
    LOCK_PATH.parent.mkdir(parents=True, exist_ok=True)
    with LOCK_PATH.open('a+') as lock:
        try:
            fcntl.flock(lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            raise RuntimeError('Another coordinator owns the GPU endpoint; wait for that batch to finish') from error
        try:
            lock.seek(0)
            lock.truncate()
            json.dump({'pid':os.getpid(),'run_name':run_name,'utc':dt.datetime.now(dt.timezone.utc).isoformat()}, lock)
            lock.flush()
            models = await preflight_endpoint(root)
            write_json(Path(run_root)/'qa/endpoint_models.json', models)
            write_json(Path(run_root)/'qa'/f'{run_name}_admission.json', admission)
            print('PARALLEL_ADMISSION', json.dumps(admission), flush=True)
            summary, records = await run_batch(jobs, admission['concurrency'], run_root,
                                               run_name, registry(root), ENDPOINT)
            # The batch wall clock includes the slowest call, not the sum of
            # overlapping calls. Cached calls are explicitly identified.
            for record in records:
                print('INSTANCE_RUNTIME', record['call_id'],
                    'INPUT_TOKENS', record['input_tokens'],
                    'OUTPUT_TOKENS', record.get('output_tokens', 'unavailable'),
                    'QUEUE_SECONDS', round(record.get('client_queue_seconds', 0), 6),
                    'GENERATION_SECONDS', round(record.get('elapsed_seconds', 0), 6),
                    'TOTAL_SECONDS', round(record.get('total_seconds', 0), 6),
                    'CACHED', bool(record.get('resumed_from_disk')), flush=True)
            if summary.get('resumed_without_generation'):
                print('HISTORICAL_BATCH_WALL_CLOCK_SECONDS', summary['historical_batch_elapsed_seconds'], flush=True)
            else:
                print('BATCH_WALL_CLOCK_SECONDS', summary['elapsed_seconds'], flush=True)
            print('CURRENT_INVOCATION_SECONDS', summary['current_invocation_seconds'], flush=True)
            return summary, records
        finally:
            fcntl.flock(lock.fileno(), fcntl.LOCK_UN)
```

### Experiment Codes/src/participant_agent_interaction.py

```python
"""Participant Agent Interaction protocol on vLLM: parallel independent pairs, sequential reviewed turns.
"""
from __future__ import annotations
from pathlib import Path
import ast
import copy
import json

from workbench_protocol import (ROOT, FrozenTokenizer, read_json, write_json, sha_text,
                                seed_for, spawn_instance, write_json_once, bind_run_sources)
from workbench_runtime import run_parallel, require_execution_permission, validate_run_id


def interaction_prompts(root=ROOT):
    root = Path(root)
    path = root/'Participant_Agent_Interaction/prompts/prompt.py'
    tree = ast.parse(path.read_text())
    debate = next(ast.literal_eval(node.value) for node in tree.body if isinstance(node, ast.Assign)
                  and any(isinstance(t, ast.Name) and t.id=='DEBATE_SYSTEM' for t in node.targets))
    return (root/'Participant_Agent_Interaction/prompts/initial_system.txt').read_text(), debate


def question_for(plan):
    return 'Regarding ' + plan['row']['context'] + ':\n' + '\n'.join(
        c.capitalize() + ': ' + plan['row']['questions'][c] for c in plan['components'])


def initial_template(parts):
    descriptions = {'magnitude':'number and unit only',
        'actor':'name, plus role or affiliation if the question asks for it',
        'mechanism':'brief description of the requested mechanism'}
    lines = []
    for c in parts:
        lines.extend([c.upper()+': <'+descriptions[c]+'>',
                      'QUOTE_'+c.upper()+': "<sentence copied word for word from YOUR article>"'])
    return '\n\nFill in this template. Replace each <...> and keep every label exactly as written:\n' + '\n'.join(lines) + '\n<END>'


def new_interaction_sessions(run_id, root=ROOT):
    validate_run_id(run_id)
    plans = read_json(Path(root)/'Participant_Agent_Interaction/inputs/session_plan.json')
    sessions = []
    for plan in plans:
        s = copy.deepcopy(plan)
        s.update(question=question_for(plan), history=[], latest={}, status='ready',
                 debate_turns=0, reviews=[], run_id=run_id,
                 agents_instances={a:spawn_instance(run_id+':'+plan['session_id']+':'+a, 'participant_'+a.lower()).__dict__ for a in plan['agents']})
        sessions.append(s)
    return sessions


def interaction_input(session, participant, stage, root=ROOT):
    """Reproduce the original Participant Agent Interaction system/user contents and one-user serialization."""
    root = Path(root)
    agent = session['agents'][participant-1]
    peer = session['agents'][2-participant]
    article = (root/'Participant_Agent_Interaction/inputs/articles'/agent/(session['article_id']+'.txt')).read_text()
    initial, debate = interaction_prompts(root)
    if stage == 'initial':
        system = initial
        user = ('Your article:\n<article>\n'+article+'\n</article>\n\nQuestion:\n'+
                session['question']+initial_template(session['components']))
    else:
        system = debate.format(agent=agent,peer=peer)
        history = '\n\n'.join('Agent '+h['agent']+', '+h['stage']+', turn '+str(h['turn'])+':\n'+h['output'] for h in session['history'])
        user = ('ASSIGNED ARTICLE\n'+article+'\n\nQUESTION\n'+session['question']+
                '\n\nDISCUSSION SO FAR\n'+history+'\n\nRespond only as Agent '+agent+'.')
    return system, user


def review_packet(session):
    return {'session_id':session['session_id'], 'question':session['question'],
            'components':session['components'], 'debate_turns':session['debate_turns'],
            'latest':session['latest'], 'history':session['history'],
            'instruction':'Review whole latest messages without consulting truth. Every requested component must have the same determinate meaning. Shared uncertainty, missing components or contradictory commitments do not establish consensus.'}


def packet_hash(packet):
    return sha_text(json.dumps(packet,ensure_ascii=False,sort_keys=True))


def apply_review(session, review):
    expected = packet_hash(review_packet(session))
    if review['packet_sha256'] != expected:
        raise ValueError('Stale review; current messages do not match')
    determinate, components = review['determinate'], review['component_agreement']
    if len(determinate)!=2 or any(type(v) is not bool for v in determinate):
        raise ValueError('Two determinacy judgments are required')
    if review['components'] != session['components'] or len(components)!=len(session['components']):
        raise ValueError('Every requested component needs a judgment')
    if any(type(v) is not bool for v in components) or type(review['agreement']) is not bool:
        raise ValueError('Use explicit Boolean judgments')
    if review['agreement'] != (all(determinate) and all(components)):
        raise ValueError('Overall agreement conflicts with component assessments')
    if not review.get('rationale','').strip() or not review.get('reviewer','').strip():
        raise ValueError('Record reviewer and substantive reason')
    session['reviews'].append(review)
    if review['agreement']:
        session.update(status='complete', stop_reason='agreement_before_debate' if not session['debate_turns'] else 'substantive_agreement')
    elif session['debate_turns'] >= 20:
        session.update(status='complete', stop_reason='ten_round_limit')
    else:
        session['status'] = 'ready'
    return session


def submit_review(run_id, session_id, *, agreement, determinate, component_agreement,
                  rationale, reviewer, inspected_packet_sha256, reviewed_debate_turn, root=ROOT):
    validate_run_id(run_id)
    folder = Path(root)/'Participant_Agent_Interaction/runs'/run_id
    session = read_json(folder/'data/sessions'/(session_id+'.json'))
    if session['status'] != 'awaiting_review':
        raise ValueError('Session is not waiting for a consensus review')
    packet = review_packet(session)
    if reviewed_debate_turn != session['debate_turns'] or inspected_packet_sha256 != packet_hash(packet):
        raise ValueError('The inspected review packet is stale; inspect the current messages')
    review = {'packet_sha256':inspected_packet_sha256, 'agreement':agreement,
        'determinate':determinate, 'components':session['components'],
        'component_agreement':component_agreement, 'rationale':rationale, 'reviewer':reviewer}
    apply_review(copy.deepcopy(session), review)  # Validate before saving.
    name = session_id+f'_t{session["debate_turns"]:02d}.json'
    path = folder/'data/review_inbox'/name
    if path.exists() and read_json(path) != review:
        raise ValueError('Review already exists; never overwrite a stopping decision')
    write_json(path, review)
    return review


async def advance_interactions(run_id, *, allow_inference=False, root=ROOT):
    """Run one ready wave; return immediately when only semantic reviews remain.

    Both initials are independent jobs. Thereafter each pair contributes at most
    one next turn per wave, and only after its preceding review has been applied.
    """
    require_execution_permission(allow_inference)
    root = Path(root)
    validate_run_id(run_id)
    folder = root/'Participant_Agent_Interaction/runs'/run_id
    sources = list((root/'Participant_Agent_Interaction/inputs').rglob('*.json'))
    sources += list((root/'Participant_Agent_Interaction/inputs/articles').rglob('*.txt'))
    sources += list((root/'Participant_Agent_Interaction/prompts').glob('*'))
    sources += list((root.parent/'src').glob('*.py'))
    sources += list((root.parents[1]/'Parallel Agent Setup/src').glob('*.py'))
    sources += [root/'config/model.json', root/'Participant_Agent_Interaction/run_interactions.py']
    sources += [root/'config/adapter_registry.json']
    bind_run_sources(folder, sources, root.parents[1])
    states = []
    for fresh in new_interaction_sessions(run_id, root):
        path = folder/'data/sessions'/(fresh['session_id']+'.json')
        s = read_json(path) if path.exists() else fresh
        if s['status']=='awaiting_review':
            review = folder/'data/review_inbox'/(s['session_id']+f'_t{s["debate_turns"]:02d}.json')
            if review.exists():
                apply_review(s, read_json(review))
                write_json(path, s)
        states.append(s)
    ready = []
    tokenizer = FrozenTokenizer(root)
    for s in states:
        if s['status']!='ready':
            continue
        if s['history']:
            if s['debate_turns'] >= 20:
                raise ValueError('Turn limit reached without a stopping review')
            turn = s['debate_turns']+1
            participant = s['first_speaker'] if turn%2 else 3-s['first_speaker']
            specs = [(participant,'debate',turn)]
        else:
            specs = [(1,'initial',0),(2,'initial',0)]
        for participant, stage, turn in specs:
            logical = f'participant-{s["session_id"]}-' + (f'initial-{participant}' if stage=='initial' else f'turn{turn:02d}')
            call_id = run_id+'_'+logical.replace('-','_')
            system, user = interaction_input(s, participant, stage, root)
            serialized, ids = tokenizer.prepare(system+'\n\n'+user)
            agent = s['agents'][participant-1]
            job = {'call_id':call_id,'kind':'participant_interaction','model':'participant_'+agent.lower(),
                'input_ids':ids,'serialized':serialized,'temperature':0.9,'max_tokens':384,
                'seed':seed_for(logical,20230930),'stop':['<END>'],'stop_token_ids':[2]}
            ready.append((s,participant,stage,turn,job))
    if not ready:
        return {'new_calls':0,'complete':sum(s['status']=='complete' for s in states),
                'awaiting_review':[s['session_id'] for s in states if s['status']=='awaiting_review']}
    jobs = [x[4] for x in ready]
    wave = 'wave_'+sha_text('|'.join(j['call_id'] for j in jobs))[:12]
    write_json_once(folder/'inputs'/(wave+'_input_texts.json'), [j['serialized'] for j in jobs])
    summary, records = await run_parallel(jobs,wave,folder,allow_inference=True,root=root)
    by_call = {r['call_id']:r for r in records}
    if any(r['status']!='complete' for r in records):
        raise RuntimeError('A Participant Agent Interaction call failed. Preserve all logs and resolve the technical failure; no automatic resampling.')
    touched = {}
    for s,participant,stage,turn,job in ready:
        r = by_call[job['call_id']]
        message = {'call_id':job['call_id'],'agent':s['agents'][participant-1],
            'participant':participant,'stage':stage,'turn':turn,'output':r['output_text'].removesuffix('</s>'),
            'input_tokens':r['input_tokens'],'output_tokens':r['output_tokens'],
            'finish_reason':r['finish_reason'],'runtime_seconds':r['elapsed_seconds']}
        s['history'].append(message)
        s['latest'][str(participant)] = message
        s['debate_turns'] = turn
        touched[s['session_id']] = s
    for sid,s in touched.items():
        s['status']='awaiting_review'
        write_json(folder/'data/sessions'/(sid+'.json'),s)
        packet = review_packet(s)
        write_json_once(folder/'data/review_pending'/(sid+f'_t{s["debate_turns"]:02d}.json'),
                        {**packet, 'packet_sha256':packet_hash(packet)})
    return {'new_calls':len(jobs),'batch':summary,
            'awaiting_review':[s['session_id'] for s in states if s['status']=='awaiting_review'],
            'complete':sum(s['status']=='complete' for s in states)}


def export_interactions(run_id, root=ROOT):
    """Export completed new sessions only; source Participant Agent Interaction data stay untouched."""
    import csv
    root = Path(root)
    validate_run_id(run_id)
    folder = root/'Participant_Agent_Interaction/runs'/run_id
    sessions = [read_json(folder/'data/sessions'/(p['session_id']+'.json'))
                for p in read_json(root/'Participant_Agent_Interaction/inputs/session_plan.json')]
    if any(s['status']!='complete' for s in sessions):
        raise ValueError('All20 sessions must have a stopping review before final export')
    dest = root/'Participant_Agent_Interaction/deliverables'/run_id
    dest.mkdir(parents=True, exist_ok=True)
    write_json(dest/'sessions.json',sessions)
    with (dest/'All_Interaction_Transcripts.txt').open('w') as f:
        for s in sessions:
            f.write(s['session_id']+'\n'+s['question']+'\n\n')
            for h in s['history']:
                f.write(f'{h["agent"]} | {h["stage"]} | turn {h["turn"]}\n{h["output"]}\n\n')
    with (dest/'Participant_Agent_Interaction_Summary.csv').open('w',newline='') as f:
        fields = ['session_id','article_id','debate_turns','stop_reason','input_tokens','output_tokens','source_accuracy']
        w = csv.DictWriter(f,fields)
        w.writeheader()
        for s in sessions:
            w.writerow({**{k:s[k] for k in fields[:4]},
                'input_tokens':sum(h['input_tokens'] for h in s['history']),
                'output_tokens':sum(h['output_tokens'] for h in s['history']),
                'source_accuracy':'requires separate source-based assessment'})
    return dest
```

### Experiment Codes/Mistral/Participant_Agent_Interaction/run_interactions.py

```python
"""Participant Agent Interaction: prepare, advance, review, status and export.

Twenty pairs run in parallel waves. Within a pair, the next debate turn waits
for an explicit semantic stopping review. This CLI never invents that judgment.
"""
from pathlib import Path
import argparse
import asyncio
import json
import sys

REPOSITORY = Path(__file__).resolve().parents[3]
sys.path[:0] = [str(REPOSITORY / "Experiment Codes/src"),
               str(REPOSITORY / "Parallel Agent Setup/src")]
from workbench_protocol import FrozenTokenizer, read_json
from workbench_runtime import validate_run_id
from participant_agent_interaction import (
    new_interaction_sessions, interaction_input, advance_interactions,
    submit_review, export_interactions,
)
WORKSPACE = REPOSITORY / "Experiment Codes/Mistral"


def prepare(run_id):
    tokenizer = FrozenTokenizer(WORKSPACE)
    sessions = new_interaction_sessions(run_id, WORKSPACE)
    sizes = []
    for session in sessions:
        for participant in (1, 2):
            system, user = interaction_input(session, participant, "initial", WORKSPACE)
            _, ids = tokenizer.prepare(system + "\n\n" + user)
            if len(ids) + 384 > 16384:
                raise ValueError("Initial context overflow: " + session["session_id"])
            sizes.append(len(ids))
    print(json.dumps({"sessions": len(sessions), "independent_initial_calls": len(sizes),
        "maximum_initial_input_tokens": max(sizes), "reserved_output_tokens": 384,
        "maximum_debate_turns_per_pair": 20, "inference_executed": False}, indent=2))
    return sessions


def status(run_id):
    validate_run_id(run_id)
    folder = WORKSPACE / "Participant_Agent_Interaction/runs" / run_id
    states = []
    for path in sorted((folder / "data/sessions").glob("*.json")):
        session = read_json(path)
        states.append({k: session.get(k) for k in ("session_id", "status", "debate_turns", "stop_reason")})
    print(json.dumps(states, indent=2))
    return states


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("action", choices=["prepare", "advance", "review", "status", "export"], nargs="?", default="prepare")
    parser.add_argument("--run-id", default="participant_interaction_001")
    parser.add_argument("--execute", action="store_true", help="Required only to send model requests with advance.")
    parser.add_argument("--review-file", type=Path, help="JSON list of actual judgments; see the local README.")
    args = parser.parse_args(argv)
    validate_run_id(args.run_id)
    if args.action == "prepare":
        prepare(args.run_id)
    elif args.action == "advance":
        if not args.execute:
            parser.error("advance requires --execute to make model calls")
        progress = asyncio.run(advance_interactions(args.run_id, allow_inference=True, root=WORKSPACE))
        print(json.dumps(progress, indent=2))
    elif args.action == "review":
        if args.review_file is None:
            parser.error("review requires --review-file")
        reviews = read_json(args.review_file)
        if not isinstance(reviews, list) or not reviews:
            raise ValueError("Provide a nonempty JSON list of completed review judgments")
        for review in reviews:
            submit_review(args.run_id, root=WORKSPACE, **review)
        print("Saved semantic reviews:", len(reviews))
    elif args.action == "status":
        status(args.run_id)
    elif args.action == "export":
        destination = export_interactions(args.run_id, root=WORKSPACE)
        print("Deliverables:", destination.relative_to(REPOSITORY))


if __name__ == "__main__":
    main()
```

## Prepare

In [ ]:
spec = importlib.util.spec_from_file_location("participant_interaction_entrypoint", ROOT / 'Experiment Codes/Mistral/Participant_Agent_Interaction/run_interactions.py')
runner = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = runner
spec.loader.exec_module(runner)
RUN_ID = "participant_interaction_001"
sessions = runner.prepare(RUN_ID)


## Advance one wave

The first wave generates40 independent initial answers. Later waves generate at most one turn per ready pair. After each wave, inspect the complete pending review packets before continuing.

In [ ]:
from participant_agent_interaction import advance_interactions, submit_review, export_interactions
work = ROOT / "Experiment Codes/Mistral"
if RUN_INFERENCE:
    progress = await advance_interactions(RUN_ID, allow_inference=True, root=work)
    print(progress)
else:
    print("No model calls; all sessions remain prepared.")


## Submit actual stopping reviews

Use the JSON schema in the adjacent README. Every judgment must be based on the current complete messages, include a reviewer and reason, and match the packet hash and debate-turn number. Agreement stops immediately; disagreement ends after at most20 debate turns. Repeat the review and advance cells until all sessions finish.

In [ ]:
REVIEWS = []  # Actual completed reviews only; no default consensus decision.
if RUN_INFERENCE and REVIEWS:
    for review in REVIEWS:
        submit_review(RUN_ID, root=work, **review)
    progress = await advance_interactions(RUN_ID, allow_inference=True, root=work)
    print(progress)


## Export completed sessions

In [ ]:
EXPORT_COMPLETED = False
if RUN_INFERENCE and EXPORT_COMPLETED:
    destination = export_interactions(RUN_ID, root=work)
    print("Exported:", destination.relative_to(ROOT))
